# 從零開始做！超簡陋 RAG :)
應該說是「疑似 RAG」的東西。  

這邊提供的步驟都是超級省略版，目的導向。  
如果想找完整的試錯過程 + 超醜但詳細解釋的話，請見 [這個資料夾](../../exp/aqing) 中，以 **_fullComment.py** 結尾的檔案。  

為什麼中英夾雜因為這個人有時候真的懶得切鍵盤，然後寫英文就是為了裝逼      
為什麼英文的語法像大便因為窩的母語就不是英文🥲  

## Outline
- Set the Environment
- Scripts that can be used Directly
- Scripts that can be used Directly
- Embedding Model for Single Sentence
    * Tokenizer 
    * Embedding Model (make token embeddings)
    * Pooling
    * Comparing Sentence Similarity
- 打包 Handmade Embedding Model: **qingEmbedding()**
- RAG!
    * Chunking
    * Make the `memoryBank` (very little-scale vector database)
    * Query (the **R** of RAG)
    * Speak like Humankind (the **G** of RAG)

## Set the Environment
### Hardware
All code in this notebook was tested on MacBook Air without 獨顯.  
The runtime is lightweight (if you use the same LM as I did.) and each cell should finish quickly (usually < 1 min),  
so 電腦的配置似乎並不是非常重要 for this tutorial.   

But in [maybe-RAG.py](../../interface/aqing/maybe-RAG.py), syntax for calling the gpu (`.to(torch.device)`) was added to allow any possibility of running the code on computers other than feifei.  

### Internet
The function `transformers.AutoModel.from_pretrained()` will automatically download the required model from [Hugging Face](https://huggingface.co).   
Therefore, you need an internet connection when running the code for the first time.  
### Code env
This tutorial is based on **Python**. It is recommended to install needes modules in a virtual environment like `conda` or something else, using either `pip install` or `conda install`.  
Required modules are listed below. You can also find detail in [reauirements.txt](requirements.txt).  

**Main Requirements:** 
- numpy
- ollama
- pathlib
- pytorch (torch)
- transformers

In [ ]:
### --------------------  Import python Models -------------------- ###
import numpy as np
from ollama import chat
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModel

## Scripts that can be used Directly
這邊列出了一些開蓋即食的東西，但必須確保：
1. `Conda env` 中列出的所有 python 套件已經透過 `pip` 或 `conda` 安裝。
2. 參考文本需存在於與腳本所在目錄平行的、名為 `dataset` 的資料夾中，結構如下：  
```
.
├── put-scripts-here/          # 腳本所在目錄
│   ├── similarityDemo.py
│   ├── queryDemo.py
│   ├── the-R-of-RAG.py
│   └── maybe-RAG.py
│ 
├── dataset/                   # 與腳本所在目錄平行的、名為 `dataset` 的資料夾
│   ├── what-are-active-galactic-nuclei.txt        
│   ├── alma-basic.txt
│   └── mvp-proposal.txt
```
推薦一個暴力解的方法，就是把整個 repo clone 下來，這樣就只需要擔心模組有沒有裝齊全的問題啦！

### 開蓋即食 :)))
[similarityDemo.py](../../exp/aqing/similarityDemo.py)  
可以 print 出兩個句子的語意相似度，句子是要從程式碼裡面單獨輸入的。  
這邊沒有用到分割文本或之類的東西，純粹在展示向量嵌入模型。  

[queryDemo.py](../../exp/aqing/queryDemo.py)  
引入了一篇文章，並且做了文本分割，可以從 `textPath` 那邊修改要模型讀的文章。  
這邊可以做到比較使用者輸入和文章中所有分段中，哪段的語意最接近，並 print 出來。  
使用者輸入 `theQuery` 需要從程式碼裡面改，對啦就是沒做介面啦。  
用 `what-are-active-galactic-nuclei.txt` 當範例文本，因為這個的效果最好 。  
(偷偷說效果最差的是 `mvp-proposal.txt`，絲毫不意外。)  

[the-R-of-RAG.py](../../interface/aqing/the-R-of-RAG.py)  
只有檢索沒有生成，基本上可以說是 [queryDemo.py](../../exp/aqing/queryDemo.py) 的附使用者介面(cli)版。  

[maybe-RAG.py](../../interface/aqing/maybe-RAG.py)  
目前的最終產品，加上了偵測 gpu 存在的語法。  
並透過 [ollama](https://ollama.com) 接上了生成式模型，所以務必記得 `pip install ollama`。  
可以選擇的解碼器語言模型有 `Llama2`, `llama3:70b` (大小寫有差)  
當然這取決於您電腦的顯存與硬碟空間，如果是自己的小筆電的話牆裂建議用 `Llama2`，  
因為 `llama3:70b` 光參數檔就有 40GB 啦哈哈

## Embedding Model for Single Sentence
這邊會有一點點解釋和教學，如果要直接看見可以產生 sentence embedding 的功能的話可以快轉到 「打包手作嵌入模型: qingEmbedding()」。  

### Tokenizer
Tokenizer 中文叫分詞器。可以把自然語言句子先分割成 token, 再轉成 tokenID.  
分割的規則取決于使用的語言模型，英文可能會有拆字跟的情況，比如複數的 's' 自己算一個 token.  
轉成 tokenID 的方法是查字典，字典 (voca) 也是語言模型自帶的。  

好欸什麼都用別人的，就這個開源爽。

In [ ]:
### ----------------------------  Text ---------------------------- ###
testString_1 = "I like astronomy."
testString_2 = "I enjoy watching the night sky full of stars."
testString_3 = "I like tomatoes."
testString_4 = 'I trust the universe will always bring me to you'
### --------------------------  Tokenizer ------------------------- ###
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
inp_1 = tokenizer(testString_1, return_tensors='pt')
inp_2 = tokenizer(testString_2, return_tensors='pt')
inp_3 = tokenizer(testString_3, return_tensors='pt') 
inp_4 = tokenizer(testString_4, return_tensors='pt')
re_tokens = tokenizer.convert_ids_to_tokens(inp_4['input_ids'][0]) # tokenID 還原成字, 超樸實的函數命名


#### Tokenizer 的一點解釋
`AutoTokenizer.from_pretrained()` 會去 [Hugging Face](https://huggingface.co) 找到並**下載**語言模型，因為 `transformers` 就是 [Hugging Face](https://huggingface.co) 發行的套件，所以記得連網。  

這邊使用的語言模型叫做 `all-MiniLM-L6-v2`，是 [Hugging Face](https://huggingface.co) 上語意判斷領域裡面最多人下載的一款模型。  
當然也可以用別的你喜歡的，但就不保證運行的需求和時間ㄌ  

`inp_*` 是將 `teatString_*` 放進分詞器分割的產物，本體是一個類似字典的資料結構。裡面包含 `'input_ids'`（就是 tokenID 本人！） 和 `'attention_mask'`，  
這兩個是有用的等下還要用。

`tokenizer(return_tensors='pt')` 的 `'pt'` 代表 pytorch，接下來的張量運算會用到火炬蟒，所以把 tensor 打包成 pytorch 能認的格式。也可以 `='tf'` for tensorflow.  

### Embedding Model (make token embeddings)
將 tokenIDs 變成 token embeddings (嵌入向量 之 詞向量)  
簡單來說是在嵌入矩陣裡面查表。  
久遠以前的做法是獨熱編碼和嵌入矩陣相乘，但因為效率的問題所以現在一般是用查表 (indexing) 的。

In [ ]:
### -----------------------  Token Embeddings ----------------------- ###
ebModel = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
torch.set_grad_enabled(False) # 訓練模型(改變)的時候才需要梯度, 只是使用模型的話不用, 所以關掉省電
out_1 = ebModel(**inp_1) # **代表全部帶入, inp 裡面有 'input_ids', 'token_type_ids' ...
out_2 = ebModel(**inp_2)
out_3 = ebModel(**inp_3)
torch.set_grad_enabled(True)

print(out_1.last_hidden_state.shape) # torch.Size([1, 6, 384]) -> (batch size, token num, embedding dim)

#### Embedding Model 的一點解釋
`AutoModel.from_pretrained()` 會去 [Hugging Face](https://huggingface.co) 找到並**下載**語言模型，和 tokenizer 一樣的邏輯。   

把 `inp_*` 丟進嵌入模型 `ebModel` 裡面，回傳值命名為 `out_*`。    

`out_*` 裡面最重要（也是唯一有用的）是 `last_hidden_state`，這神經網路最後一層的輸出，通常來說是語意完整的好東西。
他的形狀可以用 `.shape` or `.size()` 查看，print 出來的順序是 (batch size, token num, embedding dim)  
- batch size: 不知道怎樣但好像都是 1
- token num: 一個句子 (testString_*) 被分割成幾個 token
- embedding dim: 隱藏層的大小(維度, hidden_size, hidden_dim)，每個 token 經過模型後被轉換成 384 維 (this ebmodel) 的向量表示

### Pooling (make sentence embedding)
所謂池化就是把多個詞向量 (token embeddings) 壓成一個句子向量 (sentence embedding)。  
可能在卷積神經網路 (CNN) 的領域，池化又叫做降採樣，但在語意相似的領域，池化的意義好像稍微不同。  
總之，這邊我會自己定義一個池化的函式，採用的方法是**平均池化**。  

The idea of mean pooling is simple:  
Sum all token embeddings in a sentence and divide by the number of valid tokens.  
The result is a single vector that represents the sentence as a whole.  
In other words, it is literally an **average**.

From a geometric point of view, the sentence embedding is located at the average position of its constituent word vectors in the embedding space (vector space).  
For example, if a sentence contains many terms such as *“submillimeter waves”*, *“interferometer”*, *“observation time”*, and *“deep-sky objects”*, its sentence vector is likely to be closer to the region associated with *radio astronomy*.  

Of course, there are other pooling methods.  
One common example is **max pooling**, which selects the strongest token embedding as the sentence representation.  
但因為目前沒有用到所以先不說 :)

In [ ]:
def meanPooling(eb_model_output, attention_mask): 
    token_emb = eb_model_output.last_hidden_state
    atMaskP1d = attention_mask.unsqueeze(-1) # 增加一個維度, 和 token_emb 對齊 (attention mask plus 1 dim)
    atMaskP1d = atMaskP1d.expand(token_emb.shape).float() # 最後一個維度沿著 token_emb.shape 複製, 總之對齊造型
                                                          # float() 因為同是浮點數才能運算, numpy 基操
    efficient_token_emb = token_emb * atMaskP1d # 與遮罩相乘, 遮罩=1 的地方才留下值
    poolingResult = torch.sum(efficient_token_emb, dim=1) # 沿著張量相乘結果的的第1條軸(seq_len)加總, 
                                                          # 即所有真實 token 向量的總和, 耶 pooling
    poolingResult_mean = poolingResult / torch.clamp(atMaskP1d.sum(dim=1), min=1e-9) # 平均
                                         # atMaskP1d.sum(dim=1)是沿著注意力遮罩的第1條軸相加, 
                                         # 因為都是 1, 0 所以相加的值就是有效的 token 的數量
                                         # torch.clamp() 是除數!=0保護機制, 等於零的話就當作 1e-9處理
    return poolingResult_mean

#### Pooling 的一點解釋
這個函式複雜的地方主要是**對齊形狀**，因為真的麻煩的要死所以等我閒得不行的時候才會更新。

### Comparing Sentense Similarity
使用剛剛的 `meanPooling()`，將詞向量們 `out_*` 壓成句子向量 `sentence_emb_*`。  

In [ ]:
sentence_emb_1 = meanPooling(out_1, inp_1['attention_mask']) # Perform pooling, 把 N 個 token_emb 壓成一條 sentence_emb
sentence_emb_1 = torch.nn.functional.normalize(sentence_emb_1, p=2, dim=1) # Normalize embeddings
sentence_emb_2 = meanPooling(out_2, inp_2['attention_mask'])
sentence_emb_3 = meanPooling(out_3, inp_3['attention_mask'])

### ------------------------------  Similarity ------------------------------ ###  
cosSim_11 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_1)                                 
cosSim_12 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_2)
cosSim_13 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_3)
print('cosine')
print(f'string1 & string1: {cosSim_11[0]}') # 因為就是一樣的句子啊屁眼
print(f'string1 & string2: {cosSim_12[0]:.2f}')
print(f'string1 & string3: {cosSim_13[0]:.2f}') # 成功證明嘻嘻

#### Comparing Sentense Similarity 的一點解釋
還記得我們的例句是:
```
testString_1 = "I like astronomy."
testString_2 = "I enjoy watching the night sky full of stars."
testString_3 = "I like tomatoes."
```
1, 3 長得很像，用字也多有重疊；但就任費來說，1, 2 的予以才是新進的。  
所以分別 print 出 1&2, 1&3 的餘弦相似度 (越接近 1 代表越相近)，  
結果模型表現得挺盡人意！  
非常好，結案。

---

## 打包手作嵌入模型: qingEmbedding()
雖然叫做 **qings**Embedder() 有點名過其實，但隨便啦，叫什麼都好。  
打包了從分詞到平均池化的所有工作，並且可以透過 `need_normalize` 選擇是否正規化。

In [ ]:
def qingsEmbedder(theItem, need_normalize): # theStr: 要轉成embedding的東西, need_normalize(bool)
    torch.set_grad_enabled(False)
    inp = tokenizer(theItem, return_tensors="pt")
    out = ebModel(**inp)
    token_emb = out.last_hidden_state
    atMaskP1d = inp['attention_mask'].unsqueeze(-1).expand(token_emb.shape).float()
    sentenceEmbedding = torch.sum(token_emb * atMaskP1d, dim=1) / torch.clamp(atMaskP1d.sum(dim=1), min=1e-9)
    if need_normalize == True:
        sentenceEmbedding = torch.nn.functional.normalize(sentenceEmbedding, p=2, dim=1)
    elif need_normalize == False:
        pass
    else :
        print("'need_normalize' should be a boolean value.")
    torch.set_grad_enabled(True)
    return sentenceEmbedding

# Demo
sentence_emb_1 = qingsEmbedder(testString_1, True)
sentence_emb_2 = qingsEmbedder(testString_2, True)
sentence_emb_3 = qingsEmbedder(testString_3, True)
cosSim_12 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_2)
cosSim_13 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_3)
# torch.nn.functional.cosine_similarity() 也可以寫成 torch.nn.functional.cosine_sim()
# 就像 torch.matmul() 可以寫成 torch.mm() 一樣, 真的縮寫得很隨便 
print(f'string1 & string2: {cosSim_12[0]:.2f}')
print(f'string1 & string3: {cosSim_13[0]:.2f}')

---

## RAG!
終於要用到我們的 `textFile` !  

In [ ]:
### ----------------------------  Text ---------------------------- ###
root = Path(__file__).resolve().parents[2]
textPath = f'{root}/dataset/what-are-active-galactic-nuclei.txt' # from nasa
textFile = open(textPath, 'r').read() 

一點點注意的東西：  
open() 打開的是一個通往檔案的管道，要用 `.reaad()` 讀了之後才會是字串  
else: AttributeError: '_io.TextIOWrapper' object has no attribute 'split'

### Chunking
參照語言模型的 config.json, 發現他的 `max_position_embeddings = 512`  
這代表模型最多可以給出 512 個位置編碼，也就是說，一次丟進去的 token 超過 512 個就會報錯。  
所以，要讓語言模型可以處理長文的話就需要先將文章切割。

In [ ]:
### ---------------------------------  Chunking -------------------------------- ###
posEmb_max = ebModel.config.max_position_embeddings # read the config.json of ebModel
chunks = []
# This function can deal with too long chunks
def cutLong(tooLongItem, maxLen, overlap): # 與 chunks(list) 相依!!
    startIdx = 0
    while startIdx < len(tooLongItem):
        endIdx = startIdx + maxLen
        chunks.append(tooLongItem[startIdx:endIdx]) # 分割後的東東
        startIdx += overlap # 留一點重疊
    return chunks

chunkLen_max = int(np.floor(posEmb_max*0.75)) # 向下取整, 1 token ~ 0.75 word
for c in textFile.split('\n\n'):
    clean_c = c.strip() # remove space
    if len(clean_c) >= chunkLen_max:
        cutLong(clean_c, chunkLen_max, 50)
        print('Too long chunks detected, Need more time...')
    elif len(clean_c) > 30: # 長度太短的應該是廢話
        chunks.append(clean_c)

### Make the `memoryBank`
將所有片段 (儲存在 `chunks`) 塞進 `qingsEmbedder()`，變成 sentence embeddings.  
`torch.cat` for 堆疊。

In [ ]:
### -----------------------  Chunks -> Sentence Embeddings ---------------------- ###
sentenceEmbs = []
for c in chunks:
    sentenceEmbs.append(qingsEmbedder(c, True)) # normalized
    
### ----------------------------  Make the Memory Bank --------------------------- ###
memoryBank = torch.cat(sentenceEmbs, dim=0)

### Query

In [ ]:
### ---------------------------------  Try Query --------------------------------- ###
#theQuery = 'what is the different between seyfert and qusar?'
theQuery = 'is agn a blackhole?'
queryEmb = qingsEmbedder(theQuery, True)
sim_scores = torch.matmul(queryEmb, memoryBank.t()) # 已經正規化過了所以可以直接用矩陣乘法
'''# same as ""
sim_scores = queryEmb @ memoryBank.t()
'''

topScores, topIndex = torch.topk(sim_scores, k=3)
print(f'In all {len(chunks)} chunks, these are top three most simi:') # 他的的英文亂講
for i in range(len(topScores[0])):
    print(f'Similarity scores: {(topScores[0][i]):.2f}')
    print(f'--> {chunks[topIndex[0][i]]}')
    print()

### Speak like Humankind 

In [ ]:
# some user interactions
decoderName = input("Input the decoder (e.g., Llama2) >>> ") # decoder for 說人話
textName = input("Pick a file to use as reference >>> ")

In [ ]:
def decoder_OLMA(ref, usrQuery):
    respone = chat(
        model=decoderName,
        messages=[
            {'role': 'system', 
             'content': f'You are a helpful assistant. Answer the question using ONLY these information: /n{ref}'},
            {'role': 'user',
             'content': usrQuery}]
    )
    return respone['message']['content']

In [ ]:
while True:
    theQuery = input("Put your query here (or enter 'quit' to quit) >>> ")
    print()
    if theQuery=='quit':
        print('byeeee ;))')
        break
    else:
        queryEmb = qingsEmbedder(theQuery, True)
        simiScores = queryEmb @ memoryBank.t()
        topScores, topIndex = torch.topk(simiScores, k=3)

        simiContent = ''
        for i in topIndex[0]:
            simiContent += chunks[i]

        ollamaRe = decoder_OLMA(ref=simiContent, usrQuery=theQuery)
        print(ollamaRe)
        print('---------------------------------------------------------------------------------', end='\n\n')
    input("Press 'ENTER' to start a new query.")